# LiDAR 初期化 3DGS を Colab で学習する

`tools/gaussian_splatting/pack_colab_bundle.py --harmonize-exposure` で作った束（`transforms.json` + 露出を揃えた JPEG + 着色済み `init.ply`）から
`train_gsplat.py` で 3D Gaussian Splatting を学習し、学習に使わなかった view で PSNR/SSIM を測り、
指定 view を写真と並べて描画する。

1. ランタイム → ランタイムのタイプを変更 → GPU（T4 で動く設定。L4/A100 なら速い）。
2. 束を Google Drive に置き、下の `BUNDLE` を合わせる。
3. 上から順に実行する。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BUNDLE = '/content/drive/MyDrive/lidarslam_3dgs/stadtgarten_bundle.zip'  # 束の場所
OUT = '/content/drive/MyDrive/lidarslam_3dgs/out'                         # 結果の保存先
!rm -rf /content/bundle && mkdir -p /content/bundle && unzip -q "$BUNDLE" -d /content/bundle
!ls /content/bundle | head; ls /content/bundle/images | wc -l

In [ ]:
# gsplat は初回の学習時に CUDA カーネルをビルドする（数分かかる）。
!git clone -q --depth 1 -b develop https://github.com/rsasaki0109/lidar_slam_ros2.git /content/lidar_slam_ros2
!pip -q install gsplat

## 学習

- `--holdout-every 8`: 8 枚に 1 枚を学習から外し、その PSNR/SSIM を報告する（新しい視点での品質）。
- `--render-views`: 写真と並べて描画する view（transforms のフレーム番号）。Stadtgarten の 300 は
  「GEIG Bauwerksanierung」の横断幕が写る held-out view。
- `--lidar-depth-lambda`: LiDAR の深度で形状を固定し、浮遊物を抑える。
- `--mcmc --mcmc-cap`: ガウシアン数の上限。T4（16 GB）で足りなければ下げる。

In [ ]:
ITERS = 30000
RENDER_VIEWS = '300,296,304'
!mkdir -p /content/out
!cd /content/lidar_slam_ros2 && python3 tools/gaussian_splatting/train_gsplat.py \
  --transforms /content/bundle/transforms.json --init-ply /content/bundle/init.ply \
  --out /content/out/model.ply --iters {ITERS} --sh-degree 3 --knn-scale-init \
  --mcmc --mcmc-cap 2000000 --lidar-depth-lambda 0.1 --ssim-lambda 0.2 \
  --holdout-every 8 --render-views {RENDER_VIEWS} --render-dir /content/out/renders
!cat /content/out/model.ply.metrics.json

## 結果を見る

`*_compare.png` は左が 3DGS の描画、右が写真。下のセルは横断幕のあたりを切り出して拡大する
（座標は 1600x1200 画像の画素、別の場面では合わせて変える）。

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

CROP = (1000, 330, 1450, 560)  # x0, y0, x1, y1（横断幕）
for path in sorted(glob.glob('/content/out/renders/*_compare.png')):
    image = Image.open(path)
    w = image.width // 2
    render, photo = image.crop((0, 0, w, image.height)), image.crop((w, 0, image.width, image.height))
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, im, title in ((axes[0], render, '3DGS'), (axes[1], photo, 'photo')):
        ax.imshow(im.crop(CROP)); ax.set_title(f'{path.split("/")[-1]}  {title}'); ax.axis('off')
    plt.show()

In [ ]:
# 結果を Drive に残す（model.ply は SuperSplat などで開ける）
!mkdir -p "$OUT" && cp -r /content/out/* "$OUT"/ && ls -la "$OUT"